# NBA Team Rankings Using Massey Ratings

This notebook uses the **Massey ranking method** to estimate team strength from NBA game results and evaluate how well those ratings predict held-out games.

The analysis focuses on the 2025–2026 NBA regular season and uses a 15-point cap on scoring margins so that blowouts do not disproportionately influence team ratings.

**Technologies:** Python, NumPy, Pandas, KaggleHub

## 1. Imports and Data Loading

The game data comes from the Kaggle *Historical NBA Data and Player Box Scores* dataset. KaggleHub downloads the dataset, and the notebook locates `Games.csv` within the downloaded directory.

In [ ]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

In [ ]:
dataset_path = Path(
    kagglehub.dataset_download(
        "eoinamoore/historical-nba-data-and-player-box-scores"
    )
)

games_file = next(dataset_path.rglob("Games.csv"))
df = pd.read_csv(games_file, low_memory=False)
df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

print(f"Loaded {len(df):,} game records.")

## 2. Filter to the 2025–2026 Regular Season

The dataset is restricted to the 2025–2026 regular season. Rows without valid home or away team information are removed before modeling.

In [ ]:
start_date = pd.Timestamp("2025-10-20")
end_date = pd.Timestamp("2026-04-13")

season_games = df.loc[
    (df["gameDateTimeEst"] >= start_date)
    & (df["gameDateTimeEst"] < end_date)
    & df["hometeamCity"].notna()
    & df["awayteamCity"].notna()
].copy()

season_games = season_games.reset_index(drop=True)

print(f"Regular-season games: {len(season_games):,}")

## 3. Create Training and Test Sets

The first 1,000 games are used to estimate team ratings and the remaining games are held out for evaluation. Keeping the games in chronological order allows the model to be tested on later games.

In [ ]:
train_size = 1000

train_data = season_games.iloc[:train_size].copy()
test_data = season_games.iloc[train_size:].copy()

print(f"Training games: {len(train_data):,}")
print(f"Test games: {len(test_data):,}")

## 4. Define NBA Teams

Each team is assigned an index so game results can be represented in the Massey matrix.

In [ ]:
team_names = [
    "Hawks", "Celtics", "Nets", "Hornets", "Bulls", "Cavaliers",
    "Mavericks", "Nuggets", "Pistons", "Warriors", "Rockets",
    "Pacers", "Clippers", "Lakers", "Grizzlies", "Heat", "Bucks",
    "Timberwolves", "Pelicans", "Knicks", "Thunder", "Magic",
    "76ers", "Suns", "Trail Blazers", "Kings", "Spurs", "Raptors",
    "Jazz", "Wizards"
]

team_to_index = {team: i for i, team in enumerate(team_names)}

print(f"Teams included: {len(team_names)}")

## 5. Construct the Massey System

The Massey method represents team ratings with the linear system

\[
Mr = p
\]

where:

- **M** records how often teams play one another,
- **r** is the vector of unknown team ratings, and
- **p** contains cumulative point differentials.

A maximum point differential of **15 points** is used. This reduces the influence of unusually large blowouts.

In [ ]:
def build_massey_system(data, point_cap=15):
    """Construct the Massey matrix M and point-differential vector p."""
    n_teams = len(team_names)
    M = np.zeros((n_teams, n_teams), dtype=float)
    p = np.zeros(n_teams, dtype=float)

    for row in data.itertuples():
        home_idx = team_to_index[row.hometeamName]
        away_idx = team_to_index[row.awayteamName]

        M[home_idx, home_idx] += 1
        M[away_idx, away_idx] += 1
        M[home_idx, away_idx] -= 1
        M[away_idx, home_idx] -= 1

        point_diff = row.homeScore - row.awayScore
        point_diff = np.clip(point_diff, -point_cap, point_cap)

        p[home_idx] += point_diff
        p[away_idx] -= point_diff

    # Massey constraint: all team ratings sum to zero.
    M[-1, :] = 1
    p[-1] = 0

    return M, p

## 6. Calculate Massey Ratings

Solving the linear system produces one numerical strength rating for each team. Higher values indicate stronger estimated performance under the model.

In [ ]:
def calculate_massey_ratings(M, p):
    """Solve the Massey linear system and return the team-rating vector."""
    return np.linalg.solve(M, p)


M_train, p_train = build_massey_system(train_data, point_cap=15)
training_ratings = calculate_massey_ratings(M_train, p_train)

In [ ]:
rankings = (
    pd.DataFrame({
        "Team": team_names,
        "Massey Rating": training_ratings
    })
    .sort_values("Massey Rating", ascending=False)
    .reset_index(drop=True)
)

rankings.index = rankings.index + 1
rankings.index.name = "Rank"

rankings

## 7. Evaluate Game-Winner Predictions

For each game in the test set, the model predicts the team with the higher training-set Massey rating as the winner. Accuracy is the proportion of evaluated games for which the predicted winner matches the actual winner.

In [ ]:
def evaluate_predictions(data, ratings):
    """Return prediction accuracy on a set of games."""
    correct = 0
    evaluated = 0

    for row in data.itertuples():
        home_idx = team_to_index[row.hometeamName]
        away_idx = team_to_index[row.awayteamName]

        point_diff = row.homeScore - row.awayScore

        if point_diff == 0:
            continue

        actual_winner = "Home" if point_diff > 0 else "Away"
        predicted_winner = (
            "Home" if ratings[home_idx] >= ratings[away_idx] else "Away"
        )

        correct += actual_winner == predicted_winner
        evaluated += 1

    return correct / evaluated if evaluated else np.nan


test_accuracy = evaluate_predictions(test_data, training_ratings)
print(f"Test accuracy: {test_accuracy:.1%}")

## 8. Limitations and Possible Improvements

This ranking system intentionally uses a relatively simple representation of team strength. Several factors that affect real NBA outcomes are not included:

- injuries and player availability,
- trades and roster changes,
- home-court advantage,
- changes in team performance over time, and
- differences in the importance or recency of games.

A future version could weight recent games more heavily, incorporate player availability, test different point-differential caps, or evaluate the approach across multiple seasons.

## Conclusion

This project demonstrates how linear algebra can be applied to a real sports-ranking problem. NBA game results are converted into a matrix system, solved for team-strength ratings, and evaluated on games that were not used to construct those ratings.

The 15-point scoring cap provides a simple way to reduce the effect of extreme margins while retaining information about how decisively each game was won.